# requisitos
* Scikit-learn
* gensim
* modelo de vetorização do usado no pytext obtido em https://fasttext.cc/docs/en/crawl-vectors.html 
O modelo deve ser o vetor e deve ser salvo em /modelos

# Dados
Os dados utilizados foram gerados de forma sinteca utiliando LLM, foram gerados 1000 itens de cada categoria para o modelo 

# Metodologia
O modelo matematico escolhido para o modelo foi o de regressão linear pela sua simplicidade e facilidade de lidar com textos e vetores, serão usados modelos de embedding para tratar os erros de ortogratia(caso existam)

In [ ]:
import json
import pandas as pd

json_cateorias: dict | None = None
with open('seeds/seed_categorias.json', mode='r') as json_cateorias_arquivo:
    json_cateorias = json.load(json_cateorias_arquivo)


In [ ]:
linhas_df = []
for categoria, lista_descricoes in json_cateorias.items():
    for item in lista_descricoes:
        linhas_df.append({'categoria': categoria, 'descricao': item})

df_categoria = pd.DataFrame(linhas_df, columns=['categoria', 'descricao'])

# Conferindo carga dos dados

In [ ]:
print("amostra dos dados")
print(df_categoria.head())
print()

print("colunas: " + str(df_categoria.columns.tolist()))
print("Total de linhas: " + str(len(df_categoria)))
print()

print("Total de itens por categoria")
print(df_categoria['categoria'].value_counts())


# treino
hora que o filho chora e a mãe não vê

In [ ]:
from sklearn.model_selection import train_test_split

X = df_categoria['descricao']  
y = df_categoria['categoria']

X_train, X_test, y_train, y_test = train_test_split(
    X, y,
    test_size=0.2,
    random_state=42,
    stratify=y
)

## Pipeline

In [ ]:
from gensim.models import KeyedVectors

import numpy as np
from sklearn.base import BaseEstimator, TransformerMixin
from sklearn.linear_model import LogisticRegression
from sklearn.pipeline import Pipeline

class FastTextVectorizer(BaseEstimator, TransformerMixin):
    def __init__(self, model, dim=300):
        self.model = model
        self.dim = dim
    
    def fit(self, X, y=None):
        return self
    
    def transform(self, X):
        vetores = []
        for texto in X:
            palavras = texto.lower().split()
            vetores_palavras = []
            for palavra in palavras:
                if palavra in self.model:
                    vetores_palavras.append(self.model[palavra])
            
            if vetores_palavras:
                vetor = np.mean(vetores_palavras, axis=0)
            else:
                vetor = np.zeros(self.dim)
            
            vetores.append(vetor)
        
        return np.array(vetores)

# Pipeline
pipeline = Pipeline([
    ("fasttext", FastTextVectorizer(model = KeyedVectors.load_word2vec_format("modelos/cc.pt.300.vec"))),
    ("clf", LogisticRegression(max_iter=1000, C=1.0, random_state=42)),
])

sample_weights = np.ones(len(X_train))

palavras_fronteira = ['mensalidade', 'pagamento', 'conta', 'boleto', 'taxa']

for i, (desc, cat) in enumerate(zip(X_train, y_train)):
    if cat != 'MORADIA':
        if any(re.search(rf'\b{p}\b', desc.lower()) for p in palavras_fronteira):
            sample_weights[i] = 3.0

# Treinar
pipeline.fit(X_train, y_train, clf__sample_weight=sample_weights)

## Teste


In [ ]:
from sklearn.metrics import classification_report, confusion_matrix, accuracy_score

y_pred = pipeline.predict(X_test)

y_proba = pipeline.predict_proba(X_test)

print(f"Acurácia geral: {accuracy_score(y_test, y_pred):.2%}")

print("\nRelatório por categoria:")
print(classification_report(y_test, y_pred))

print("\nMatriz de confusão:")
print("(Linhas = real | Colunas = previsto)\n")
cm = confusion_matrix(y_test, y_pred, labels=pipeline.classes_)
cm_df = pd.DataFrame(cm, index=pipeline.classes_, columns=pipeline.classes_)
print(cm_df)

print("\n" + "="*60)
print("ERROS DO MODELO:")
print("="*60)

erros = X_test[y_test != y_pred]
reais = y_test[y_test != y_pred]
previstos = y_pred[y_test != y_pred]
confiancas = y_proba[y_test != y_pred].max(axis=1)

for desc, real, prev, conf in zip(erros, reais, previstos, confiancas):
    print(f'\n  "{desc}"')
    print(f'    Real:      {real}')
    print(f'    Previsto:  {prev}')
    print(f'    Confiança: {conf:.1%}')


print("\n" + "="*60)
print("CONFIANÇA DAS PREVISÕES:")
print("="*60)

confiancas_gerais = y_proba.max(axis=1)
print(f"Média de confiança:   {confiancas_gerais.mean():.1%}")
print(f"Menor confiança:      {confiancas_gerais.min():.1%}")
print(f"Maior confiança:      {confiancas_gerais.max():.1%}")

# Quantas previsões o modelo fez com pouca confiança (< 50%)?
baixa_confianca = (confiancas_gerais < 0.5).sum()
print(f"\nPrevisões com < 50% de confiança: {baixa_confianca} de {len(y_test)}")

In [ ]:
def classificar_despesa(descricao: str, pipeline, limite_confianca: float = 0.70):
    """
    Classifica uma descrição de despesa.
    Se a confiança for menor que o limite, retorna 'OUTROS'.
    """
    # Prever categoria e probabilidades
    categoria = pipeline.predict([descricao])[0]
    probabilidades = pipeline.predict_proba([descricao])[0]
    
    # Pegar a confiança da categoria prevista
    confianca = probabilidades.max()
    
    # Se não tiver confiança suficiente, manda para "OUTROS"
    if confianca < limite_confianca:
        return "OUTROS", confianca
    
    return categoria, confianca

descricao = 'Uber'
print(pipeline.predict([descricao]))
print(
    pipeline.predict_proba([descricao])[0].max()
)
